Retreive 20 records including the last 4 columns from the GCS's file generated from ex01.ipynb
Using "gemini-2.5-flash", ask it to summarize the reports.

- Pre-requisite : Generate API keys through [Google AI Studio](https://aistudio.google.com/app/apikey)

In [1]:
import json
import os

from dotenv import load_dotenv
from google import genai
from google.oauth2 import service_account
from google.cloud import storage

In [2]:
load_dotenv(dotenv_path='../.env')

True

In [3]:
genai_api_key = os.getenv("GEMINI_API_KEY")
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
file_name = f"sf_police_report/2025-09-08.json"

In [4]:
def retrieve_data_from_gcs(service_account_key: str,
                           project_id: str,
                           bucket_name: str,
                           file_name: str,
                           key_list: list
                           ) -> list:
    credentials = service_account.Credentials.from_service_account_file(service_account_key)
    client = storage.Client(project=project_id,
                            credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    content = json.loads(file.download_as_string())

    output = []
    for data in content:
        row = []
        
        for key in key_list:
            row.append(data.get(key, None))
        output.append(row)
    return output

In [5]:
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
data = retrieve_data_from_gcs(service_account_key, 
                              project_id,
                              bucket_name,
                              file_name,
                              key_list)

In [6]:
filtered_data = [row[-4:] for row in data][:20]

In [ ]:
# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=genai_api_key)

In [8]:
model_name = "gemini-2.5-flash"

In [9]:
prompt_content = f"There has been a police report\
on the following list of [description, lon, lat, district] : {filtered_data} recently.\
Summarize the reports"

In [10]:
response = client.models.generate_content(
    model=model_name,
    contents=prompt_content
)

In [11]:
response.text

'Here\'s a summary of the police reports:\n\nA total of **20 police reports** were filed, covering a diverse range of incidents.\n\n**Key Categories and Noteworthy Incidents:**\n\n*   **Property Crimes (8 reports):** This category includes various **theft incidents** (including two from unlocked vehicles over $950, and one for $50-$200), a **residence burglary** (unlawful entry), and several **vehicle-related crimes** (two stolen vehicles—an auto and a motorcycle—and one recovered auto, which occurred "Out of SF").\n*   **Violent & Assaultive Crimes (5 reports):** Reports include **battery** (one with serious injuries), **robbery** (two incidents, one a street/public place robbery with force), and one **aggravated assault with force**.\n*   **Public Order & Miscellaneous (7 reports):** This broad category covers incidents like **false personation**, a **suspicious occurrence**, a **violation of a restraining order**, a **municipal police code violation**, two **narcotics-related offens